# Project 63 — Local RAG A/B Testing
## Compare Retrieval Strategies Side-by-Side

**Stack:** LangChain · ChromaDB · Ollama · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb rank-bm25

## Step 1 — Build Test Corpus

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
import shutil

llm = ChatOllama(model="qwen3:8b", temperature=0.1)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

docs = [
    Document(page_content="Machine learning uses algorithms to learn patterns from data. "
             "Supervised learning requires labeled training data. Common algorithms include "
             "linear regression, decision trees, and neural networks.", metadata={"source": "ml_intro"}),
    Document(page_content="Deep learning is a subset of machine learning using neural networks "
             "with many layers. CNNs excel at image tasks. RNNs and transformers handle sequences. "
             "Attention mechanisms revolutionized NLP.", metadata={"source": "deep_learning"}),
    Document(page_content="Reinforcement learning trains agents through rewards and penalties. "
             "Q-learning and policy gradient are key approaches. Applications include game playing, "
             "robotics, and recommendation systems.", metadata={"source": "rl"}),
    Document(page_content="Transfer learning uses pre-trained models on new tasks. Fine-tuning "
             "adjusts weights for specific domains. Few-shot and zero-shot learning minimize data needs. "
             "Foundation models like GPT demonstrate emergent capabilities.", metadata={"source": "transfer"}),
    Document(page_content="MLOps covers the operational aspects of ML. It includes data versioning, "
             "experiment tracking, model serving, and monitoring. Tools include MLflow, DVC, and Kubeflow. "
             "CI/CD for ML automates model deployment.", metadata={"source": "mlops"}),
]
print(f"Corpus: {len(docs)} documents")

## Step 2 — Strategy A: Small Chunks + Dense Retrieval

In [ ]:
splitter_a = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks_a = splitter_a.split_documents(docs)

shutil.rmtree("chroma_a", ignore_errors=True)
store_a = Chroma.from_documents(chunks_a, embeddings, persist_directory="chroma_a")
retriever_a = store_a.as_retriever(search_kwargs={"k": 3})
print(f"Strategy A: {len(chunks_a)} small chunks (100 chars)")

## Step 3 — Strategy B: Large Chunks + Dense Retrieval

In [ ]:
splitter_b = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks_b = splitter_b.split_documents(docs)

shutil.rmtree("chroma_b", ignore_errors=True)
store_b = Chroma.from_documents(chunks_b, embeddings, persist_directory="chroma_b")
retriever_b = store_b.as_retriever(search_kwargs={"k": 3})
print(f"Strategy B: {len(chunks_b)} large chunks (300 chars)")

## Step 4 — A/B Comparison

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import time

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based ONLY on the context. If unsure, say so."),
    ("human", "Context: {context}\n\nQuestion: {question}")
])
chain = qa_prompt | llm | StrOutputParser()

test_questions = [
    "What is transfer learning?",
    "What tools are used in MLOps?",
    "How do CNNs differ from RNNs?",
]

results = []
for q in test_questions:
    for label, retriever in [("A-small", retriever_a), ("B-large", retriever_b)]:
        start = time.time()
        docs_found = retriever.invoke(q)
        context = "\n".join([d.page_content for d in docs_found])
        answer = chain.invoke({"context": context, "question": q})
        elapsed = time.time() - start
        results.append({
            "strategy": label, "question": q[:30],
            "chunks_found": len(docs_found),
            "context_len": len(context),
            "answer_len": len(answer),
            "latency": round(elapsed, 2),
        })
        print(f"  {label} | {q[:30]} | {len(docs_found)} chunks | {elapsed:.1f}s")

import pandas as pd
rdf = pd.DataFrame(results)
print("\nA/B COMPARISON")
print(rdf.groupby("strategy")[["context_len","answer_len","latency"]].mean().round(2).to_string())

## What You Learned
- **A/B testing** for retrieval strategies
- **Chunk size impact** on context and answer quality
- **Quantitative comparison** framework for RAG